# Week 05A — Features and Scale Space

**MCTA 4364 Machine Vision** · Session 1 of Week 5 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W05_features/W05A_features_scale_space.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W05_features/W05A_features_scale_space.ipynb)

Segmentation (Week 4) describes *regions*. Many tasks need to find **the same physical point** in different images instead: aligning a part
to a reference, stitching panoramas, tracking, measuring 3D, localising a robot. The answer is **local features**: small, distinctive image
patches (**keypoints**) with a numerical **descriptor**, that can be re-detected when the viewpoint, scale, rotation or lighting changes.

### Learning outcomes
By the end of this session you will be able to:
1. Explain why simple **template matching** fails under rotation and scale, and what **invariance** means.
2. Build **Gaussian and Laplacian pyramids** and use them for multi-scale processing and blending.
3. Describe **scale space**, the Laplacian/Difference-of-Gaussian blob detector and automatic **scale selection**.
4. Derive the **Harris** and **Shi–Tomasi** corner detectors from the structure tensor, and measure invariance with a **matching-score** benchmark.
5. Explain how **SIFT** and **ORB** achieve scale and rotation invariance, and compare float vs binary descriptors.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1 | Why local features? Template matching and its limits |
| 10–25 | 2 | Image pyramids, Laplacian pyramid, pyramid blending |
| 25–40 | 3 | Scale space and blob detection |
| 40–58 | 4 | Corners: structure tensor, Harris, Shi–Tomasi, FAST (interactive) |
| 58–72 | 5–6 | SIFT and ORB; invariance benchmark |
| 72–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import skimage.data
from skimage.feature import blob_log

from cvhelpers import SMOKE, check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
box = load_sample("box.png", gray=True)
scene = load_sample("box_in_scene.png", gray=True)
building = load_sample("building.jpg")
graf1 = load_sample("graf1.png")
apple, orange = load_sample("apple.jpg"), load_sample("orange.jpg")
show(box, scene, building, graf1, titles=["box.png (object)", "box_in_scene.png", "building.jpg", "graf1.png"], figsize=(22, 4.5))

## 1. Why local features?

The simplest way to find an object is **template matching**: slide the object image over the scene and compute a similarity score at every
position, e.g. the normalised cross-correlation (`cv2.matchTemplate` with `TM_CCOEFF_NORMED`). It is excellent when the object appears with
the **same size, orientation and lighting** (e.g. a fixtured part), but fails as soon as the object is rotated, scaled, tilted or partly hidden.

Good local features are
- **repeatable**: found again at the same physical point after the image changes,
- **distinctive**: their descriptor is different from those of other points,
- **local**: small, so they survive occlusion and clutter,
- **invariant** (or robust) to rotation, scale, brightness and moderate viewpoint change,
- **efficient** and **numerous**: enough points, computed quickly.

In [ ]:
res_direct = cv2.matchTemplate(scene, box, cv2.TM_CCOEFF_NORMED)
rotated_scene = cv2.warpAffine(scene, cv2.getRotationMatrix2D((scene.shape[1] / 2, scene.shape[0] / 2), 30, 0.8), scene.shape[::-1])
res_rot = cv2.matchTemplate(rotated_scene, box, cv2.TM_CCOEFF_NORMED)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.4))
for k, (sc, res, name) in enumerate([(scene, res_direct, "original scene"), (rotated_scene, res_rot, "scene rotated 30 deg, scaled 0.8")]):
    _, mx, _, loc = cv2.minMaxLoc(res)
    vis = cv2.cvtColor(sc, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(vis, loc, (loc[0] + box.shape[1], loc[1] + box.shape[0]), (0, 0, 255), 3)
    axes[2 * k].imshow(vis[..., ::-1]); axes[2 * k].set_title(f"{name}: best match (score {mx:.2f})"); axes[2 * k].axis("off")
    im_ = axes[2 * k + 1].imshow(res, cmap="magma"); axes[2 * k + 1].axis("off")
    axes[2 * k + 1].set_title(f"correlation map (range {res.min():.2f} to {res.max():.2f})")
    plt.colorbar(im_, ax=axes[2 * k + 1], fraction=0.035)
plt.suptitle("Figure 1.1 - Template matching: a partial match in the original scene, a wrong match after rotation and scaling", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Image pyramids

A **Gaussian pyramid** is a stack of progressively blurred and half-sized images (`cv2.pyrDown`: Gaussian blur, then drop every other row and
column). Processing each level with the *same* small operator is equivalent to using larger and larger operators on the original, which makes
it the cheapest way to handle **scale**: coarse-to-fine search, multi-scale detection, optical flow (Week 10) and CNN feature pyramids (Week 8).

A **Laplacian pyramid** stores what each Gaussian level loses: $L_i = G_i - \text{up}(G_{i+1})$. It is a band-pass decomposition. Adding the
levels back from coarse to fine **reconstructs the image exactly**. This makes it possible to blend two images band by band, so the seam is
smooth at low frequencies but sharp at high ones (Burt & Adelson, 1983).

In [ ]:
def gaussian_pyramid(img, levels):
    pyr = [img]
    for _ in range(levels):
        pyr.append(cv2.pyrDown(pyr[-1]))
    return pyr


def laplacian_pyramid_demo(img, levels):
    g = gaussian_pyramid(img.astype(np.float32), levels)
    lap = [g[i] - cv2.pyrUp(g[i + 1], dstsize=g[i].shape[1::-1]) for i in range(levels)]
    return lap + [g[-1]]


gpyr = gaussian_pyramid(building, 4)
lpyr = laplacian_pyramid_demo(cv2.cvtColor(building, cv2.COLOR_BGR2GRAY), 4)
fig = plt.figure(figsize=(22, 10))
x = 0.0
for k, g in enumerate(gpyr):
    w = g.shape[1] / building.shape[1] * 0.3
    ax = fig.add_axes([x, 0.5, w, w * building.shape[0] / building.shape[1] * 22 / 10])
    ax.imshow(cv2.cvtColor(g, cv2.COLOR_BGR2RGB)); ax.set_title(f"G{k}: {g.shape[1]}x{g.shape[0]}", fontsize=9); ax.axis("off")
    x += w + 0.01
x = 0.0
for k, l in enumerate(lpyr):
    w = l.shape[1] / building.shape[1] * 0.3
    ax = fig.add_axes([x, 0.0, w, w * building.shape[0] / building.shape[1] * 22 / 10])
    ax.imshow(l, cmap="gray" if k == len(lpyr) - 1 else "seismic", vmin=None if k == len(lpyr) - 1 else -40, vmax=None if k == len(lpyr) - 1 else 40)
    ax.set_title(f"L{k}" if k < len(lpyr) - 1 else f"G{k} (residual)", fontsize=9); ax.axis("off")
    x += w + 0.01
fig.suptitle("Figure 2.1 - Gaussian pyramid (top) and Laplacian pyramid (bottom: detail lost at each level)", fontweight="bold", y=1.0)
plt.show()

In [ ]:
def pyramid_blend(a, b, levels=5):
    """Blend the left half of a with the right half of b, band by band."""
    la, lb = laplacian_pyramid_demo(a, levels), laplacian_pyramid_demo(b, levels)
    blended = []
    for pa, pb in zip(la, lb):
        mask = np.zeros(pa.shape[:2], np.float32); mask[:, : pa.shape[1] // 2] = 1
        mask = mask[..., None] if pa.ndim == 3 else mask
        blended.append(pa * mask + pb * (1 - mask))
    out = blended[-1]
    for lev in reversed(blended[:-1]):
        out = cv2.pyrUp(out, dstsize=lev.shape[1::-1]) + lev
    return np.clip(out, 0, 255).astype(np.uint8)


a_, o_ = cv2.resize(apple, (512, 512)), cv2.resize(orange, (512, 512))
naive = np.hstack([a_[:, :256], o_[:, 256:]])
show(a_, o_, naive, pyramid_blend(a_, o_, 6), titles=["apple", "orange", "naive cut: visible seam", "Laplacian-pyramid blend: 'orapple'"],
     figsize=(22, 5.2))

## 3. Scale space and blob detection

**Scale space** is the family of blurred images $L(x, y, \sigma) = G_\sigma * I$: as $\sigma$ grows, structures smaller than about $\sigma$
disappear. The **scale-normalised Laplacian of Gaussian** $\sigma^2 \nabla^2 L$ responds most strongly to a **blob** of radius $r$ at
$\sigma = r/\sqrt{2}$. Searching for extrema over **position and scale** therefore finds blobs *and* their size, the principle of **automatic
scale selection** (Lindeberg, 1998). SIFT approximates the LoG by the cheaper **Difference of Gaussians**: $\text{DoG} = L(\sigma_{k+1}) - L(\sigma_k) \approx (k - 1)\,\sigma^2\nabla^2 L$.

In [ ]:
canvas = np.zeros((200, 600), np.float32)
radii = [6, 12, 24, 40]
for k, r in enumerate(radii):
    cv2.circle(canvas, (70 + 150 * k, 100), r, 1.0, -1)
sigmas = np.geomspace(2, 40, 40)
resp = []
for s in sigmas:
    log = s ** 2 * cv2.Laplacian(cv2.GaussianBlur(canvas, (0, 0), s), cv2.CV_32F)
    resp.append([-log[100, 70 + 150 * k] for k in range(4)])
resp = np.array(resp)
fig, axes = plt.subplots(1, 2, figsize=(18, 4.3), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].imshow(canvas, cmap="gray"); axes[0].set_title("4 blobs of radius 6, 12, 24, 40"); axes[0].axis("off")
for k, r in enumerate(radii):
    axes[1].plot(sigmas, resp[:, k], lw=2.5, label=f"r = {r}")
    axes[1].axvline(r / np.sqrt(2), ls=":", color=plt.cm.tab10(k))
axes[1].set_xscale("log"); axes[1].set_xlabel("sigma (log)"); axes[1].set_ylabel("-sigma^2 LoG at blob centre")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_title("each blob's response peaks at sigma = r / sqrt(2) (dotted)")
plt.suptitle("Figure 3.1 - Automatic scale selection with the scale-normalised Laplacian", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
x = np.linspace(-10, 10, 400)
g = lambda s: np.exp(-x ** 2 / (2 * s ** 2)) / (np.sqrt(2 * np.pi) * s)
s = 2.0
log1d = (x ** 2 - s ** 2) / s ** 4 * g(s)
dog = g(s * 1.6) - g(s)
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
axes[0].plot(x, log1d / np.abs(log1d).max(), lw=2.5, label="Laplacian of Gaussian (scaled)")
axes[0].plot(x, dog / np.abs(dog).max(), "--", lw=2.5, label="Difference of Gaussians, k = 1.6 (scaled)")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("DoG closely approximates LoG ('Mexican hat')")
blobs_img = load_sample("detect_blob.png", gray=True)[8:-8, 8:-8]   # crop the thin white frame of the sample image
found = blob_log(blobs_img, min_sigma=4, max_sigma=40, num_sigma=15, threshold=0.2, overlap=0.3)
axes[1].imshow(blobs_img, cmap="gray")
for y_, x_, s_ in found:
    axes[1].add_patch(plt.Circle((x_, y_), s_ * np.sqrt(2), fill=False, ec="#d93025", lw=2))
axes[1].set_title(f"LoG blob detector: {len(found)} blobs with their sizes"); axes[1].axis("off")
plt.suptitle("Figure 3.2 - DoG vs LoG, and multi-scale blob detection on detect_blob.png", fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Corners: the structure tensor

Shift a small window by $(u, v)$ and measure how much it changes: $E(u, v) = \sum_{x,y} w(x, y)\,[I(x+u, y+v) - I(x, y)]^2$.
- In a **flat** region nothing changes in any direction.
- Along an **edge**, moving *along* the edge changes nothing.
- At a **corner**, every shift changes the window: the location is well defined.

A first-order Taylor expansion gives $E(u, v) \approx [u\ v]\, M\, [u\ v]^\top$ with the **structure tensor**

$$ M = \sum_{x,y} w(x, y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix} $$

Its eigenvalues $\lambda_1, \lambda_2$ describe how much the image changes in the two principal directions: both small = flat, one large = edge,
both large = corner. **Harris** (1988) avoids computing eigenvalues with $R = \det M - k\,(\operatorname{tr} M)^2 = \lambda_1\lambda_2 - k(\lambda_1 + \lambda_2)^2$
($k \approx 0.04$–$0.06$); **Shi–Tomasi** (1994) uses $\min(\lambda_1, \lambda_2)$ and is behind `goodFeaturesToTrack` (Week 10 tracks these points).

In [ ]:
blox = load_sample("blox.jpg", gray=True).astype(np.float32)
# pick one patch of each kind automatically from the Harris response (away from the border)
_ix, _iy = cv2.Sobel(blox, cv2.CV_32F, 1, 0), cv2.Sobel(blox, cv2.CV_32F, 0, 1)
_a, _b, _c = [cv2.GaussianBlur(a, (0, 0), 3) for a in (_ix * _ix, _iy * _iy, _ix * _iy)]
_R = _a * _b - _c ** 2 - 0.05 * (_a + _b) ** 2
_inner = np.zeros_like(_R, bool); _inner[20:-20, 20:-20] = True
_corner = np.unravel_index(np.argmax(np.where(_inner, _R, -np.inf)), _R.shape)
_edge = np.unravel_index(np.argmin(np.where(_inner, _R, np.inf)), _R.shape)
_flat = np.unravel_index(np.argmin(np.where(_inner, _a + _b, np.inf)), _R.shape)
patches = {"flat": tuple(map(int, _flat)), "edge": tuple(map(int, _edge)), "corner": tuple(map(int, _corner))}
fig = plt.figure(figsize=(21, 4.8))
for k, (name, (py, px)) in enumerate(patches.items()):
    ax = fig.add_subplot(1, 4, k + 1, projection="3d")
    us = np.arange(-6, 7)
    E = np.array([[np.sum((blox[py + v - 7:py + v + 8, px + u - 7:px + u + 8] - blox[py - 7:py + 8, px - 7:px + 8]) ** 2)
                   for u in us] for v in us])
    U, V = np.meshgrid(us, us)
    ax.plot_surface(U, V, E, cmap="viridis"); ax.set_title(f"E(u, v) at a {name} patch"); ax.set_xlabel("u"); ax.set_ylabel("v")
ax = fig.add_subplot(1, 4, 4)
ax.imshow(blox, cmap="gray")
for name, (py, px) in patches.items():
    ax.add_patch(plt.Rectangle((px - 7, py - 7), 15, 15, fill=False, ec="#d93025", lw=2)); ax.text(px + 9, py, name, color="#d93025")
ax.axis("off"); ax.set_title("blox.jpg with the three patches")
plt.suptitle("Figure 4.1 - Self-similarity surfaces: flat = flat bowl, edge = valley, corner = sharp bowl", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
Ix = cv2.Sobel(blox, cv2.CV_32F, 1, 0, ksize=3); Iy = cv2.Sobel(blox, cv2.CV_32F, 0, 1, ksize=3)
Sxx, Syy, Sxy = [cv2.GaussianBlur(a, (0, 0), 1.5) for a in (Ix * Ix, Iy * Iy, Ix * Iy)]
tr, det = Sxx + Syy, Sxx * Syy - Sxy ** 2
disc = np.sqrt(np.maximum((tr / 2) ** 2 - det, 0))
l1, l2 = tr / 2 + disc, tr / 2 - disc
sel = np.random.default_rng(0).choice(blox.size, 6000, replace=False)
harris = det - 0.05 * tr ** 2
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
cls = np.where(harris.ravel()[sel] > 0.01 * harris.max(), 2, np.where(l1.ravel()[sel] > 0.05 * l1.max(), 1, 0))
axes[0].scatter(l2.ravel()[sel], l1.ravel()[sel], c=cls, cmap=plt.matplotlib.colors.ListedColormap(["#9aa0a6", "#1a73e8", "#d93025"]), s=4)
axes[0].set_xscale("symlog"); axes[0].set_yscale("symlog"); axes[0].set_xlabel("lambda 2 (smaller)"); axes[0].set_ylabel("lambda 1 (larger)")
axes[0].set_title("eigenvalue plane: grey flat, blue edge, red corner")
r_disp = np.clip(harris / (0.01 * harris.max()), -1, 1)   # saturate at 1% of the maximum so weak responses stay visible
axes[1].imshow(r_disp, cmap="seismic", vmin=-1, vmax=1); axes[1].set_title("Harris R: red > 0 corner, blue < 0 edge, white ~ 0 flat")
vis = cv2.cvtColor(blox.astype(np.uint8), cv2.COLOR_GRAY2BGR)
pts = cv2.goodFeaturesToTrack(blox.astype(np.uint8), 60, 0.02, 8)
for p in pts.reshape(-1, 2).astype(int):
    cv2.circle(vis, tuple(map(int, p)), 4, (0, 0, 255), 2)
axes[2].imshow(vis[..., ::-1]); axes[2].set_title(f"Shi-Tomasi (goodFeaturesToTrack): {len(pts)} corners")
for ax in axes[1:]:
    ax.axis("off")
plt.suptitle("Figure 4.2 - The structure tensor classifies flat, edge and corner pixels", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: Harris parameters
A low threshold marks texture and noise; a large window (`block`) blurs nearby corners together; `k` trades corner vs edge sensitivity.

In [ ]:
bgray = cv2.cvtColor(building, cv2.COLOR_BGR2GRAY)


def harris_demo(threshold_pct=1.0, block=2, k=0.04):
    R = cv2.cornerHarris(np.float32(bgray), block, 3, k)
    peaks = (R == cv2.dilate(R, np.ones((7, 7)))) & (R > threshold_pct / 100 * R.max())     # non-maximum suppression
    vis = building.copy()
    for y, x in np.argwhere(peaks):
        cv2.circle(vis, (int(x), int(y)), 3, (0, 0, 255), -1)
    show(vis, titles=[f"Harris: {peaks.sum()} corners (threshold {threshold_pct}% of max, block {block}, k {k})"], figsize=(9, 7))


interact(harris_demo, threshold_pct=widgets.FloatSlider(value=1.0, min=0.05, max=20, step=0.05),
         block=widgets.IntSlider(value=2, min=2, max=10), k=widgets.FloatSlider(value=0.04, min=0.02, max=0.2, step=0.01))

**FAST** (Rosten & Drummond, 2006) is even quicker: a pixel is a corner if $n$ contiguous pixels on a 16-pixel circle around it are all brighter
(or all darker) than the centre by a threshold. It needs only comparisons, so it runs at hundreds of frames per second on embedded CPUs and is
the detector inside ORB.

## 5. Scale- and rotation-invariant features: SIFT and ORB

**SIFT** (Lowe, 2004) combines everything above:
1. **Detect** extrema of the DoG in $(x, y, \sigma)$ across a pyramid: each keypoint has a position *and* a scale.
2. **Refine** to sub-pixel/sub-scale accuracy; reject low-contrast points and edge responses (using the Hessian, like Harris).
3. Assign an **orientation**: the dominant gradient direction in the neighbourhood. Rotating the descriptor to it gives rotation invariance.
4. **Describe** a 16×16 window (scaled to the keypoint size) as a 4×4 grid of 8-bin gradient-orientation histograms = **128 numbers**, normalised
   for illumination robustness.

**ORB** (Rublee et al., 2011) = oriented **FAST** keypoints on a pyramid + rotated **BRIEF**: a **256-bit binary** descriptor built from 256 pairwise
intensity comparisons. Binary descriptors are compared with the **Hamming distance** (XOR + bit count), which is extremely fast.

| | SIFT | ORB | AKAZE |
|---|---|---|---|
| detector | DoG extrema (scale space) | oriented FAST on a pyramid | non-linear scale space |
| descriptor | 128 floats | 256 bits | binary (MLDB) |
| matching distance | L2 | Hamming | Hamming |
| speed | moderate | very fast | fast |
| typical use | accuracy, stitching, 3D | real time, embedded, SLAM | good balance |

> **OpenCV 5 note:** `cv2.SIFT_create` and `cv2.ORB_create` are in the main package. AKAZE, BRISK and KAZE moved to `opencv-contrib-python` in OpenCV 5, so the code below only uses AKAZE when `hasattr(cv2, "AKAZE_create")`.

In [ ]:
g1 = cv2.cvtColor(graf1, cv2.COLOR_BGR2GRAY)
sift, orb = cv2.SIFT_create(nfeatures=400), cv2.ORB_create(nfeatures=400)
kp_s, des_s = sift.detectAndCompute(g1, None)
kp_o, des_o = orb.detectAndCompute(g1, None)
def draw_rich(img, kps, color, n=120):
    """Like cv2.drawKeypoints(..., DRAW_RICH_KEYPOINTS) but with thicker lines so the circles are visible in the figure."""
    vis = img.copy()
    for k in sorted(kps, key=lambda k: -k.response)[:n]:
        c, r = np.array(k.pt), max(k.size / 2, 3)
        cv2.circle(vis, tuple(map(int, c)), int(r), color, 2, cv2.LINE_AA)
        e = c + r * np.array([np.cos(np.deg2rad(k.angle)), np.sin(np.deg2rad(k.angle))])
        cv2.line(vis, tuple(map(int, c)), tuple(map(int, e)), color, 2, cv2.LINE_AA)
    return vis


vis_s = draw_rich(graf1, kp_s, (0, 255, 255))
vis_o = draw_rich(graf1, kp_o, (255, 80, 0))
show(vis_s, vis_o, titles=[f"SIFT: circle size = scale, radius line = orientation (120 strongest of {len(kp_s)})",
                          f"ORB (120 strongest of {len(kp_o)})"], figsize=(20, 7))
print("SIFT descriptor:", des_s.shape, des_s.dtype, "| ORB descriptor:", des_o.shape, des_o.dtype, "(32 bytes = 256 bits)")

In [ ]:
# Figure 5.2: the SIFT descriptor of one keypoint, drawn as a 4 x 4 grid of orientation histograms
kp = max(kp_s, key=lambda k: k.size)
S = int(kp.size * 1.5)
x0, y0 = int(kp.pt[0]) - S // 2, int(kp.pt[1]) - S // 2
patch = g1[max(y0, 0):y0 + S, max(x0, 0):x0 + S]
d = des_s[kp_s.index(kp)].reshape(4, 4, 8)
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6), gridspec_kw={"width_ratios": [1, 1, 1.3]})
axes[0].imshow(cv2.cvtColor(cv2.drawKeypoints(graf1, [kp], None, (0, 0, 255), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS),
                            cv2.COLOR_BGR2RGB)); axes[0].set_title(f"largest keypoint: size {kp.size:.0f}, angle {kp.angle:.0f} deg"); axes[0].axis("off")
axes[1].imshow(patch, cmap="gray"); axes[1].set_title("its neighbourhood"); axes[1].axis("off")
ax = axes[2]; ax.set_xlim(0, 4); ax.set_ylim(4, 0); ax.set_aspect("equal")
ang = np.arange(8) * np.pi / 4
for i in range(4):
    for j in range(4):
        ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, ec="#9aa0a6"))
        h = d[i, j] / (d.max() + 1e-6) * 0.45
        for a, v in zip(ang, h):
            ax.arrow(j + 0.5, i + 0.5, v * np.cos(a), -v * np.sin(a), head_width=0.04, color="#1a73e8")
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("descriptor: 4 x 4 cells x 8 orientation bins = 128 numbers")
plt.suptitle("Figure 5.2 - Anatomy of a SIFT descriptor", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Measuring invariance: matching score

To measure invariance, we warp `building.jpg` with a **known** rotation or scale. We then detect and describe features in both images and match them
(brute force with cross-check). Because we know the true transform, we can map every match back and count how many land within 3 pixels of where they should be. The
fraction of correct matches is the **matching score**, a variant of the evaluation protocol of Mikolajczyk & Schmid (2005).

To isolate the role of **scale space**, we compare ORB with a **single pyramid level** (`nlevels=1`) against the default ORB (8 levels) and SIFT.

In [ ]:
def matching_score(feat, norm, img, M, tol=3.0):
    h, w = img.shape
    warped = cv2.warpAffine(img, M, (w, h))
    k1, d1 = feat.detectAndCompute(img, None)
    k2, d2 = feat.detectAndCompute(warped, None)
    if d1 is None or d2 is None or len(k1) == 0 or len(k2) == 0:
        return 0.0
    m = cv2.BFMatcher(norm, crossCheck=True).match(d1, d2)
    if not m:
        return 0.0
    p1 = cv2.transform(np.float32([k1[x.queryIdx].pt for x in m])[None], M)[0]   # where the match SHOULD be
    p2 = np.float32([k2[x.trainIdx].pt for x in m])                               # where it was found
    return float((np.linalg.norm(p1 - p2, axis=1) < tol).mean())


features = {"ORB, 1 pyramid level (no scale space)": (cv2.ORB_create(500, nlevels=1), cv2.NORM_HAMMING),
            "ORB, 8 pyramid levels": (cv2.ORB_create(500), cv2.NORM_HAMMING),
            "SIFT (DoG scale space)": (cv2.SIFT_create(500), cv2.NORM_L2)}
centre = (bgray.shape[1] / 2, bgray.shape[0] / 2)
angles = [0, 15, 30, 45, 60, 90] if not SMOKE else [0, 45]
scales = [1.0, 0.8, 0.6, 0.5, 0.4] if not SMOKE else [1.0, 0.6]
colors = ["#9aa0a6", "#188038", "#d93025"]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.4))
for (name, (f, norm)), col in zip(features.items(), colors):
    axes[0].plot(angles, [matching_score(f, norm, bgray, cv2.getRotationMatrix2D(centre, a, 1.0)) for a in angles], "o-", lw=2, c=col, label=name)
    axes[1].plot(scales, [matching_score(f, norm, bgray, cv2.getRotationMatrix2D(centre, 0, s)) for s in scales], "o-", lw=2, c=col, label=name)
axes[0].set_xlabel("rotation (deg)"); axes[1].set_xlabel("scale factor"); axes[1].invert_xaxis()
for ax in axes:
    ax.set_ylabel("fraction of correct matches"); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_title("orientation assignment makes all three rotation invariant")
axes[1].set_title("without scale space, matching collapses once the scale changes")
plt.suptitle("Figure 6.1 - Matching score under known rotations and scalings", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# AKAZE lives in the main package in OpenCV 4.x but moved to opencv-contrib in OpenCV 5, so test for it
methods = {"FAST detect": lambda: cv2.FastFeatureDetector_create(30).detect(bgray),
           "ORB detect+describe": lambda: cv2.ORB_create(1000).detectAndCompute(bgray, None),
           "SIFT detect+describe": lambda: cv2.SIFT_create(1000).detectAndCompute(bgray, None)}
if hasattr(cv2, "AKAZE_create"):
    methods["AKAZE detect+describe"] = lambda: cv2.AKAZE_create().detectAndCompute(bgray, None)
times = {}
for name, fn in methods.items():
    t0 = time.perf_counter()
    for _ in range(3):
        fn()
    times[name] = (time.perf_counter() - t0) / 3 * 1000
plt.figure(figsize=(10, 3.2)); plt.barh(list(times), list(times.values()), color=["#9aa0a6", "#188038", "#d93025", "#1a73e8"][:len(times)])
plt.xlabel(f"milliseconds on this machine ({bgray.shape[1]} x {bgray.shape[0]} image)")
plt.title("Figure 6.2 - Speed of detectors and descriptors", fontweight="bold"); plt.show()

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Harris response from scratch
Return the Harris response $R = \det M - k (\operatorname{tr} M)^2$ for a float32 greyscale image, with Sobel (ksize 3) derivatives and a Gaussian window of standard deviation `sigma`. Do not call `cv2.cornerHarris`.

In [ ]:
def harris_response(img, k=0.05, sigma=1.5):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    board = np.kron((np.indices((6, 6)).sum(0) % 2), np.ones((20, 20))).astype(np.float32) * 255
    R = np.asarray(harris_response(board))
    assert R.shape == board.shape, f"expected shape {board.shape}, got {R.shape}"
    corners = [(20 * i, 20 * j) for i in range(1, 6) for j in range(1, 6)]
    at_corner = np.mean([R[y - 2:y + 3, x - 2:x + 3].max() for y, x in corners])
    assert at_corner > 0, "R should be positive at chessboard corners"
    assert R[10, 10] == 0 or abs(R[10, 10]) < 1e-3 * at_corner, "R should be ~0 in flat regions"
    assert R[10, 20] < 0, "R should be negative on an edge"


check("E1 harris_response", _test_e1)

### E2. Laplacian pyramid
Return a list `[L0, L1, ..., L(levels-1), G_levels]` for a float32 greyscale image, where $L_i = G_i - \text{pyrUp}(G_{i+1})$ (use `dstsize` so the sizes match).

In [ ]:
def laplacian_pyramid(img, levels):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    img = bgray.astype(np.float32)
    pyr = laplacian_pyramid(img, 4)
    assert len(pyr) == 5, f"expected 5 arrays (4 Laplacian levels + residual), got {len(pyr)}"
    assert pyr[0].shape == img.shape, "L0 must have the size of the input"
    ref = laplacian_pyramid_demo(img, 4)
    for k, (a, b) in enumerate(zip(pyr, ref)):
        assert a.shape == b.shape and np.allclose(a, b, atol=1e-3), f"level {k} differs from the expected Laplacian pyramid"


check("E2 laplacian_pyramid", _test_e2)

### E3. Reconstruction
Invert E2: rebuild the image from a Laplacian pyramid by repeatedly up-sampling and adding the next finer level.

In [ ]:
def reconstruct(pyr):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    img = bgray.astype(np.float32)
    rec = np.asarray(reconstruct(laplacian_pyramid_demo(img, 5)))
    assert rec.shape == img.shape, f"expected shape {img.shape}, got {rec.shape}"
    err = np.abs(rec - img).max()
    assert err < 1e-2, f"reconstruction should be exact (float), max error {err:.3g}"


check("E3 reconstruct", _test_e3)

### E4. Brightness robustness (no self-check)
Extend the matching-score benchmark of Section 6 with **brightness and contrast changes** (e.g. `cv2.convertScaleAbs(img, alpha, beta)`)
and with **Gaussian noise**. Hint: copy `matching_score` so that the second image is a photometrically changed copy and `M` is the identity
`np.float32([[1, 0, 0], [0, 1, 0]])`. Which feature is most robust to each change? Explain using how each detector and descriptor is defined.

In [ ]:
# E4: your code here

## 8. Challenge: a multi-scale template matcher

Make the template matcher of Section 1 find `box.png` in the rotated/scaled scene by searching over a **pyramid of scales** and a set of **rotations**
of the template (keep the best normalised correlation). Measure the run time and compare it with the feature-based approach in Session B.
When would you still choose template matching in a factory?

In [ ]:
# Your challenge code here

## 9. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Why is a corner a better feature than a point on a straight edge?',
        "options": [
            'It is brighter',
            'Its position is well defined in two directions, so it can be localised and re-detected precisely',
            'Edges cannot be detected',
            'Corners are always larger',
        ],
        "answer": '37a481cbed',
        "explain": 'Along an edge, shifts parallel to it change nothing.',
    },
    {
        "q": 'In the Harris detector, what do two large eigenvalues of the structure tensor indicate?',
        "options": [
            'A flat region',
            'An edge',
            'A corner',
            'Noise',
        ],
        "answer": '9b9ece098c',
        "explain": 'The intensity changes strongly in both principal directions.',
    },
    {
        "q": 'Which property makes SIFT keypoints scale-invariant?',
        "options": [
            'The 128-D descriptor',
            'Detecting extrema of the DoG over position AND scale',
            'Using colour',
            'Binary comparisons',
        ],
        "answer": '869ee0781c',
        "explain": 'Each keypoint gets its own characteristic scale.',
    },
    {
        "q": 'How is rotation invariance achieved in SIFT and ORB?',
        "options": [
            'By ignoring gradients',
            "By rotating the descriptor to the keypoint's dominant orientation",
            'By using circles only',
            'It is not achieved',
        ],
        "answer": '63f7733898',
        "explain": "The descriptor is computed in the keypoint's own orientation frame.",
    },
    {
        "q": 'ORB descriptors are compared with which distance?',
        "options": [
            'Euclidean (L2)',
            'Hamming',
            'Cosine',
            'Manhattan (L1)',
        ],
        "answer": '07e83feff3',
        "explain": 'They are 256-bit binary strings: XOR and count the bits.',
    },
    {
        "q": 'What does a Laplacian pyramid level contain?',
        "options": [
            'A smaller copy of the image',
            'The detail (band-pass) lost between two Gaussian levels',
            'The image edges only',
            'The histogram',
        ],
        "answer": '555c11d2ee',
        "explain": 'L_i = G_i - up(G_{i+1}); summing them reconstructs the image.',
    },
    {
        "q": 'A blob of radius 20 px gives the strongest scale-normalised LoG response at about which sigma?',
        "options": [
            '5',
            '14',
            '20',
            '40',
        ],
        "answer": '9d7715651d',
        "explain": 'sigma = r / sqrt(2) = 14.1.',
    },
], title='Week 05A quiz')

## 10. Exit ticket (reflection)
1. Your parts always arrive in the same orientation under a fixed camera. Would you use template matching or SIFT to locate them? Why?
2. Why does Harris fail when the image is scaled down by 2, but SIFT does not?
3. Give one reason to choose ORB over SIFT on a robot, and one reason to choose SIFT.

## 11. What's new (2025–26)

> - **Learned local features** such as SuperPoint, DISK and ALIKED, matched with **LightGlue**, now outperform SIFT under large viewpoint and
>   lighting changes and are standard in 3D reconstruction pipelines (Week 6B previews them).
> - **Dense matchers** (LoFTR, RoMa) skip keypoints entirely and match every pixel; 3D foundation models (Week 12) combine matching and geometry.
> - SIFT's patent expired in 2020, so it is part of the main OpenCV package (`cv2.SIFT_create`), and it remains a robust, well-understood baseline.

## 12. References and further exploration

**Seminal papers**
- Lowe, D. G. (2004). *Distinctive image features from scale-invariant keypoints* (SIFT). IJCV 60(2). [doi:10.1023/B:VISI.0000029664.99615.94](https://doi.org/10.1023/B:VISI.0000029664.99615.94)
- Harris, C. & Stephens, M. (1988). *A combined corner and edge detector.* Alvey Vision Conference. [doi:10.5244/C.2.23](https://doi.org/10.5244/C.2.23)
- Shi, J. & Tomasi, C. (1994). *Good features to track.* CVPR. [doi:10.1109/CVPR.1994.323794](https://doi.org/10.1109/CVPR.1994.323794)
- Rosten, E. & Drummond, T. (2006). *Machine learning for high-speed corner detection* (FAST). ECCV. [doi:10.1007/11744023_34](https://doi.org/10.1007/11744023_34)
- Rublee, E. et al. (2011). *ORB: An efficient alternative to SIFT or SURF.* ICCV. [doi:10.1109/ICCV.2011.6126544](https://doi.org/10.1109/ICCV.2011.6126544)
- Burt, P. J. & Adelson, E. H. (1983). *The Laplacian pyramid as a compact image code.* IEEE Trans. Comm. 31(4). [doi:10.1109/TCOM.1983.1095851](https://doi.org/10.1109/TCOM.1983.1095851)
- Lindeberg, T. (1998). *Feature detection with automatic scale selection.* IJCV 30(2). [doi:10.1023/A:1008045108935](https://doi.org/10.1023/A:1008045108935)
- Mikolajczyk, K. & Schmid, C. (2005). *A performance evaluation of local descriptors.* IEEE TPAMI 27(10). [doi:10.1109/TPAMI.2005.188](https://doi.org/10.1109/TPAMI.2005.188)
- DeTone, D. et al. (2018). *SuperPoint: Self-supervised interest point detection and description.* [arXiv:1712.07629](https://arxiv.org/abs/1712.07629)

**Textbooks and courses**
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §7.1 (points and patches). [szeliski.org/Book](https://szeliski.org/Book/)
- Nayar, S. *First Principles of Computer Vision*: "Feature detection" series (Harris, SIFT). [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/)
- OpenCV tutorials: *Feature detection and description*. [docs.opencv.org](https://docs.opencv.org/4.x/db/d27/tutorial_py_table_of_contents_feature2d.html)
- VLFeat: *SIFT detector and descriptor* documentation (excellent illustrations). [vlfeat.org](https://www.vlfeat.org/overview/sift.html)

**Images:** `box.png`, `box_in_scene.png`, `building.jpg`, `graf1.png`, `blox.jpg`, `detect_blob.png`, `apple.jpg`, `orange.jpg` from the OpenCV sample data.

## 13. Key takeaways
- Template matching only works without rotation/scale change; **local features** give repeatable, distinctive, invariant points.
- **Pyramids** handle scale cheaply; the **Laplacian pyramid** is a perfect, band-pass decomposition (e.g. seamless blending).
- **Scale space** + the scale-normalised LoG/DoG find blobs *and their size*.
- The **structure tensor** classifies flat/edge/corner: **Harris**, **Shi–Tomasi**; **FAST** is the fast alternative.
- **SIFT** (scale space + orientation + 128-D histogram) and **ORB** (FAST + binary BRIEF) are invariant to rotation and scale; measure the **matching score** under the changes you expect before choosing.

**Next session (05B):** matching descriptors, rejecting outliers with the ratio test and **RANSAC**, estimating homographies and **stitching panoramas**.